In [11]:
import pandas as pd
import os
from tqdm import tqdm
import re

In [12]:
def clean_text(text):
    # 1. Specifically target <br />, <br>, or <br/> and replace with a space
    # The 'i' in re.I makes it case-insensitive
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.I)
    
    # 2. Remove any other remaining HTML tags like <div> or <a>
    text = re.sub(r'<.*?>', ' ', text)
    
    # 3. Remove non-alphabetic characters (punctuation, numbers)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # 4. Convert to lowercase and fix whitespace
    text = text.lower().split()
    text = " ".join(text)
    
    return text

In [13]:
def process_and_save_folder(base_path, dataset_type):
    data = []
    for sentiment in ['pos', 'neg']:
        folder_path = os.path.join(base_path, dataset_type, sentiment)
        label = 1 if sentiment == 'pos' else 0
        files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
        
        print(f"Cleaning {dataset_type} {sentiment}...")
        for filename in tqdm(files):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_text = f.read()
                cleaned_text = clean_text(raw_text)
                data.append([cleaned_text, label])
                
    df = pd.DataFrame(data, columns=['review', 'sentiment'])
    filename_to_save = f'imdb_{dataset_type}_clean.csv'
    df.to_csv(filename_to_save, index=False)
    print(f"Saved: {filename_to_save}")

In [14]:
path = './archive/aclImdb'
process_and_save_folder(path, 'train')
process_and_save_folder(path, 'test')

Cleaning train pos...


100%|██████████| 12500/12500 [01:16<00:00, 162.69it/s]


Cleaning train neg...


100%|██████████| 12500/12500 [01:16<00:00, 164.43it/s]


Saved: imdb_train_clean.csv
Cleaning test pos...


100%|██████████| 12500/12500 [01:12<00:00, 171.78it/s]


Cleaning test neg...


100%|██████████| 12500/12500 [01:12<00:00, 172.44it/s]


Saved: imdb_test_clean.csv


In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [16]:
# 1. Load the separate clean datasets
print("Loading clean datasets...")
train_df = pd.read_csv('imdb_train_clean.csv').dropna(subset=['review'])
test_df = pd.read_csv('imdb_test_clean.csv').dropna(subset=['review'])

# 2. Assign Features (X) and Labels (y)
X_train = train_df['review']
y_train = train_df['sentiment']

X_test = test_df['review']
y_test = test_df['sentiment']

# 3. Vectorization (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)

print("Vectorizing text data...")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test) # Crucial: only transform the test data!

# 4. Train the Model
print("Training Logistic Regression model...")
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# 5. Evaluate on the official test set
y_pred = model.predict(X_test_tfidf)

print(f"\nOfficial Dataset Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Loading clean datasets...
Vectorizing text data...
Training Logistic Regression model...

Official Dataset Accuracy: 88.32%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [17]:
def predict_custom_review(text_to_test):
    # 1. Clean the input text using our regex logic
    cleaned_input = clean_text(text_to_test)
    
    # 2. Transform the text into numbers using the ALREADY TRAINED vectorizer
    # We wrap it in [cleaned_input] because the vectorizer expects a list/array
    vectorized_input = vectorizer.transform([cleaned_input])
    
    # 3. Predict using the trained model
    prediction = model.predict(vectorized_input)
    
    # 4. Convert the 1 or 0 back to a readable text
    sentiment_result = "Positive" if prediction[0] == 1 else "Negative"
    
    return sentiment_result

# --- TESTING ZONE ---
print("\n--- Testing Custom Reviews ---")

review_1 = "This movie was an absolute masterpiece! The acting was incredible and the plot kept me hooked."
print(f"Review 1: '{review_1}'")
print(f"Predicted Sentiment: {predict_custom_review(review_1)}\n")

review_2 = "Honestly, it was a complete waste of time. The script makes no sense and the ending was awful."
print(f"Review 2: '{review_2}'")
print(f"Predicted Sentiment: {predict_custom_review(review_2)}\n")

# Write your own review here to test it!
my_own_review = "The special effects were good, but overall the movie felt a bit boring and too long."
print(f"Your Review: '{my_own_review}'")
print(f"Predicted Sentiment: {predict_custom_review(my_own_review)}")


--- Testing Custom Reviews ---
Review 1: 'This movie was an absolute masterpiece! The acting was incredible and the plot kept me hooked.'
Predicted Sentiment: Positive

Review 2: 'Honestly, it was a complete waste of time. The script makes no sense and the ending was awful.'
Predicted Sentiment: Negative

Your Review: 'The special effects were good, but overall the movie felt a bit boring and too long.'
Predicted Sentiment: Negative
